[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C63_ML_System_Design_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（七步框架 / 45 分钟时间预算 / 失败模式检测 / 不确定性传播）

目标：把「system design 面试怎么答」从一堆经验之谈，变成**几个可以运行、可以断言的小工具**。

本 notebook 你会亲手实现：
1. **环境自检** —— 确认 Python / numpy 可用（本课全程不需要 GPU、不需要联网）
2. **七步框架检查器** —— 给一份「你实际讲了什么」的步骤序列，自动指出漏了哪步、哪两步顺序反了
3. **45 分钟时间预算模拟器** —— 按总时长自动缩放，给出不可妥协的检查点
4. **不确定性传播的数值演示** —— 为什么 ML 流水线的可靠性是「链式相乘」而不是「取最短板」
5. **四道练习**：失败模式检测器 / 超时报警器 / 反解「需要多可靠的单级」/ 一份完整的赛后复盘工具

> 心智模型：**评的不是画得多复杂，是「需求澄清 + 取舍显式 + 能自我批判」。**

## 0 · 环境自检

本课全程只用标准库 + numpy。没有 GPU 依赖、不联网、不下载数据。

In [ ]:
import sys, math
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert hasattr(np, 'isclose')
print('\n✅ 环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · 七步框架检查器

给一份「你实际做了什么」的动作序列（`trace`，元素是七个步骤 key 的列表，可重复、可乱序），
自动指出 **(a) 漏了哪一步？(b) 哪两步的顺序反了？**

顺序违规的定义：规范里 `a` 应在 `b` 之前，但你**首次**做 `b` 的时刻早于首次做 `a` —— 记一次违规 `(b, a)`。
这和 C62-00 的六步协议检查器是同一套逻辑，换成了七步框架。

In [ ]:
STEPS7 = [
    ('clarify_scope',   '① 澄清与范围',   '用户/场景/成功定义/失败定义/失败代价/约束/不做什么'),
    ('metrics',         '② 成功指标',     '业务指标 -> 模型指标 -> 系统指标 三层映射'),
    ('data',            '③ 数据',         '来源/标注/划分/长尾/冷启动'),
    ('modeling',        '④ 建模',         'baseline 优先，约束驱动选型'),
    ('evaluation',      '⑤ 评测',         '离线切片 + 在线 A/B，一致性检查'),
    ('serving',         '⑥ 服务与部署',   '延迟预算/容量估算/降级方案'),
    ('iteration_risk',  '⑦ 迭代与风险',   '监控/回滚/下一轮迭代计划'),
]
RANK = {k: i for i, (k, _, _) in enumerate(STEPS7)}

def check_framework(trace):
    """trace: 实际步骤序列（key 列表）。返回 (missing, inversions)。"""
    missing = [k for k, _, _ in STEPS7 if k not in trace]
    first = {}
    for pos, t in enumerate(trace):
        first.setdefault(t, pos)
    inversions = set()
    present = [k for k in RANK if k in first]
    for a in present:
        for b in present:
            if RANK[a] < RANK[b] and first[a] > first[b]:
                inversions.add((b, a))
    return missing, sorted(inversions, key=lambda p: (RANK[p[0]], RANK[p[1]]))

for k, name, what in STEPS7:
    print(f'{name:<10} {what}')

In [ ]:
# —— 三种典型表现 ——
good  = ['clarify_scope', 'metrics', 'data', 'modeling', 'evaluation', 'serving', 'iteration_risk']
jump  = ['modeling', 'clarify_scope', 'metrics', 'data', 'evaluation', 'serving', 'iteration_risk']  # 上来先讲模型
noeval = ['clarify_scope', 'metrics', 'data', 'modeling', 'serving', 'iteration_risk']               # 完全没提评测

m1, i1 = check_framework(good)
assert m1 == [] and i1 == []

m2, i2 = check_framework(jump)
assert m2 == []
assert ('modeling', 'clarify_scope') in i2 and ('modeling', 'metrics') in i2

m3, i3 = check_framework(noeval)
assert m3 == ['evaluation'] and i3 == []

print('good   -> 缺失', m1, '违规', i1)
print('jump   -> 违规', i2, '  <- 模型讲在了澄清与指标之前')
print('noeval -> 缺失', m3, '  <- 完全没有评测环节')
print('\n✅ 检查器就位：把一次设计演练的动作记录喂进来，就能自动指出漏了哪一格。')

## 2 · 45 分钟时间预算模拟器

预算按比例缩放（30/45/60 分钟通用），最后一项 `buffer` 吸收取整误差，保证**总和严格等于总时长**。
基准（45 分钟制）：①8 ②5 ③6 ④8 ⑤6 ⑥6 ⑦4 + 缓冲2。

In [ ]:
BUDGET7 = [('clarify_scope', 8), ('metrics', 5), ('data', 6), ('modeling', 8),
           ('evaluation', 6), ('serving', 6), ('iteration_risk', 4), ('buffer', 2)]   # 基准：45 分钟

def budget(total_min=45):
    """按比例缩放到 total_min，最后一项吸收取整误差。"""
    base = sum(m for _, m in BUDGET7)
    out, acc = [], 0
    for name, m in BUDGET7[:-1]:
        v = int(round(total_min * m / base))
        out.append((name, v)); acc += v
    out.append((BUDGET7[-1][0], total_min - acc))
    return out

def cumulative(total_min=45):
    """每一步「应该在第几分钟前结束」。"""
    acc, out = 0, []
    for name, m in budget(total_min):
        acc += m
        out.append((name, acc))
    return out

for total in (30, 45, 60):
    plan = budget(total)
    assert sum(v for _, v in plan) == total, (total, plan)
    print(f'{total} 分钟 :', ' '.join(f'{k}={v}' for k, v in plan))

c45 = dict(cumulative(45))
assert c45['clarify_scope'] == 8 and c45['modeling'] == 27 and c45['serving'] == 39
print('\n三个不可妥协的检查点（45 分钟制）：')
print(f"  第 {c45['clarify_scope']:>2} 分钟 —— 必须已说出「用户是谁 / 成功是什么 / 不做什么」")
print(f"  第 {c45['modeling']:>2} 分钟 —— 必须已经过完数据与建模，进入评测")
print(f"  第 {c45['serving']:>2} 分钟 —— 必须讲到服务部署，留时间给「迭代与风险」")
print('\n✅ 预算器就位。')

## 3 · 不确定性传播：为什么可靠性是「链式相乘」

传统后端工程师的直觉是「最短板决定整体」（weakest link）。ML 流水线里每一级都有独立犯错概率，
正确的模型是**相乘**，不是**取最小值**——这正是本课与传统后端 system design 的关键差异（见正文 §5）。

In [ ]:
def pipeline_success_rate(rates):
    """多级流水线的端到端正确率 = 各级独立正确率之积。"""
    r = 1.0
    for x in rates:
        r *= x
    return r

# TSR 三级流水线：检测 0.95、跨帧关联 0.90、多传感融合 0.97
tsr_rates = [0.95, 0.90, 0.97]
tsr_chain = pipeline_success_rate(tsr_rates)
tsr_weakest_link = min(tsr_rates)

assert abs(tsr_chain - 0.82935) < 1e-6
assert tsr_chain < tsr_weakest_link          # 链式相乘 < 最短板估计 —— 这是关键结论

print(f'各级正确率: {tsr_rates}')
print(f'最短板直觉给出的估计: {tsr_weakest_link:.3f}')
print(f'实际链式相乘的整体正确率: {tsr_chain:.3f}   <- 比任何单一环节都差 {(tsr_weakest_link-tsr_chain)*100:.1f} 个百分点')

# 更极端的例子：5 级流水线，每级都高达 97%（听起来已经很优秀）
five_stage = pipeline_success_rate([0.97] * 5)
assert abs(five_stage - 0.97 ** 5) < 1e-9
print(f'\n5 级流水线、每级 97%: 整体只剩 {five_stage:.3f}')
print('✅ 结论：级数越多，"每级都很优秀"造成的整体可靠性错觉就越危险。')

## ✏️ 练习 1：失败模式检测器

在 `check_framework` 基础上实现 `detect_failure_modes(trace)`，
返回触发的失败模式集合（`set`），取值来自 `{'jump_to_model', 'skip_constraints', 'skip_evaluation', 'skip_rollback'}`：

- `jump_to_model`：`('modeling','clarify_scope')` 或 `('modeling','metrics')` 出现在 inversions 里；
  **或** `'modeling'` 在 trace 里但 `'clarify_scope'` / `'metrics'` 整个缺失（完全没做，比顺序反了更严重）
- `skip_constraints`：`'clarify_scope'` 缺失
- `skip_evaluation`：`'evaluation'` 缺失
- `skip_rollback`：`'iteration_risk'` 缺失

In [ ]:
def detect_failure_modes(trace):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
onlymodel = ['modeling']   # 只做了建模，什么都没澄清

assert detect_failure_modes(good) == set()
assert detect_failure_modes(jump) == {'jump_to_model'}
assert detect_failure_modes(noeval) == {'skip_evaluation'}
assert detect_failure_modes(onlymodel) == {'jump_to_model', 'skip_constraints', 'skip_evaluation', 'skip_rollback'}

for name, trace in [('good', good), ('jump', jump), ('noeval', noeval), ('onlymodel', onlymodel)]:
    print(f'{name:<10} -> {detect_failure_modes(trace)}')
print('\n✅ 练习 1 通过：把一次模拟设计演练的动作序列喂进来，就能自动列出触发了哪些失败模式。')

## ✏️ 练习 2：超时报警器

实现 `time_budget_alarm(elapsed, done_steps, total=45)`，返回 `'ok'` / `'speed_up'` / `'fallback'`。

规则：
- `expected` = `cumulative(total)` 里 `done_steps` 最后一个元素对应的结束时刻（`done_steps` 为空则为 0）
- `elapsed <= expected + 3` → `'ok'`
- `elapsed <= expected + 10` → `'speed_up'`
- 否则 → `'fallback'`（放弃深入某一步的细节，立刻压缩讲完剩余步骤）

In [ ]:
def time_budget_alarm(elapsed, done_steps, total=45):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 45 分钟制的累计结束时刻：clarify=8 metrics=13 data=19 modeling=27 evaluation=33 serving=39 iteration_risk=43
assert time_budget_alarm(0,  []) == 'ok'
assert time_budget_alarm(10, ['clarify_scope']) == 'ok'                          # 8+3=11
assert time_budget_alarm(15, ['clarify_scope']) == 'speed_up'                    # <= 8+10=18
assert time_budget_alarm(20, ['clarify_scope']) == 'fallback'                    # > 18
assert time_budget_alarm(30, ['clarify_scope', 'metrics', 'data', 'modeling']) == 'ok'         # 27+3=30
assert time_budget_alarm(35, ['clarify_scope', 'metrics', 'data', 'modeling']) == 'speed_up'   # <= 27+10=37
assert time_budget_alarm(40, ['clarify_scope', 'metrics', 'data', 'modeling']) == 'fallback'    # > 37
assert time_budget_alarm(11, ['clarify_scope'], total=30) == 'speed_up'          # 30 分钟制：clarify 只到第 5 分钟

for e, d in [(0, []), (10, ['clarify_scope']), (20, ['clarify_scope']), (35, ['clarify_scope', 'metrics', 'data', 'modeling'])]:
    print(f'第 {e:>2} 分钟，已完成到 {d[-1] if d else "(未开始)"} -> {time_budget_alarm(e, d)}')
print('\n✅ 练习 2 通过：第 39 分钟还没讲到服务部署，报警器应该已经在叫了。')

## ✏️ 练习 3：反解「单级需要多可靠」

实现 `required_stage_reliability(k, target)`：给定流水线有 `k` 级、且假设每级正确率相同 `r`，
求满足 $r^k \ge \text{target}$ 的**最小** `r`（用公式 $r = \text{target}^{1/k}$ 直接求解即可，不需要二分）。

In [ ]:
def required_stage_reliability(k, target):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r5 = required_stage_reliability(5, 0.99)
r10 = required_stage_reliability(10, 0.99)

assert abs(r5 - 0.99 ** (1/5)) < 1e-9
assert abs(r5 ** 5 - 0.99) < 1e-9
assert abs(required_stage_reliability(1, 0.9) - 0.9) < 1e-12
assert r10 > r5, '流水线级数越多，单级需要的可靠性反而越高（同样的整体目标下）'

print(f'5 级流水线要达到整体 99%，单级至少要 {r5:.4f}')
print(f'10 级流水线要达到整体 99%，单级至少要 {r10:.4f}（比 5 级更苛刻）')
print('\n✅ 练习 3 通过：这是 §5 结论的反向用法——流水线越长，对每一级的可靠性要求越苛刻。')

## ✏️ 练习 4：赛后复盘工具（综合练习）

把前面三个工具组合成一份「45 分钟结束后」的复盘报告。
实现 `design_review(trace, elapsed_total, total=45)`，返回：

```
{'missing': [...], 'inversions': [...], 'failure_modes': {...}, 'over_time': True/False}
```

`over_time` 表示实际用时 `elapsed_total` 是否超过了 `total`（不需要用到 `time_budget_alarm`，
这是赛后的整体复盘，不是过程中的实时报警）。

In [ ]:
def design_review(trace, elapsed_total, total=45):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
r1 = design_review(good, 44, 45)
assert r1 == {'missing': [], 'inversions': [], 'failure_modes': set(), 'over_time': False}

r2 = design_review(good, 50, 45)
assert r2['over_time'] is True

r3 = design_review(noeval, 45, 45)
assert r3['missing'] == ['evaluation'] and r3['failure_modes'] == {'skip_evaluation'} and r3['over_time'] is False

r4 = design_review(onlymodel, 10, 45)
assert r4['failure_modes'] == {'jump_to_model', 'skip_constraints', 'skip_evaluation', 'skip_rollback'}

for name, r in [('good(44min)', r1), ('good(50min,超时)', r2), ('noeval', r3), ('onlymodel', r4)]:
    print(f'{name:<18} -> {r}')
print('\n✅ 练习 4 通过：一份可以直接拿去给自己模拟面试录像打分的复盘工具。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def detect_failure_modes(trace):
    missing, inversions = check_framework(trace)
    modes = set()
    jumped = (('modeling', 'clarify_scope') in inversions
              or ('modeling', 'metrics') in inversions
              or ('modeling' in trace and ('clarify_scope' in missing or 'metrics' in missing)))
    if jumped:
        modes.add('jump_to_model')
    if 'clarify_scope' in missing:
        modes.add('skip_constraints')
    if 'evaluation' in missing:
        modes.add('skip_evaluation')
    if 'iteration_risk' in missing:
        modes.add('skip_rollback')
    return modes

In [ ]:
# 练习 2 参考答案
def time_budget_alarm(elapsed, done_steps, total=45):
    cum = dict(cumulative(total))
    expected = cum[done_steps[-1]] if done_steps else 0
    if elapsed <= expected + 3:
        return 'ok'
    if elapsed <= expected + 10:
        return 'speed_up'
    return 'fallback'

In [ ]:
# 练习 3 参考答案
def required_stage_reliability(k, target):
    return target ** (1.0 / k)

In [ ]:
# 练习 4 参考答案
def design_review(trace, elapsed_total, total=45):
    missing, inversions = check_framework(trace)
    modes = detect_failure_modes(trace)
    return {
        'missing': missing,
        'inversions': inversions,
        'failure_modes': modes,
        'over_time': elapsed_total > total,
    }

---
## 🧪 真实工程胶囊：45 分钟开场脚手架 + 七步自检清单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 开场 90 秒该说什么（拿到题目后，先说这一段，不要沉默）
# ══════════════════════════════════════════════════════════════════════
# 「在我开始设计之前，我想先确认几件事：这个系统的用户是谁、在什么场景下用、
#  怎么算成功、怎么算失败、失败的代价有多大、有哪些硬约束（延迟/算力/数据）、
#  以及——同样重要——第一版我们明确不做什么。」
# 这一段话同时覆盖了七步框架的 ①，也提前打消了 §4 失败模式 ①②的风险。

# ══════════════════════════════════════════════════════════════════════
# B. 七步自检清单（面试进行中，随时对照这张表看漏了哪一格）
# ══════════════════════════════════════════════════════════════════════
# □ ① 澄清与范围   —— 用户/场景/成功/失败/代价/约束/不做什么 都说了吗？
# □ ② 成功指标     —— 业务指标 -> 模型指标 -> 系统指标，三层都提到了吗？
# □ ③ 数据         —— 来源/标注/划分/长尾，至少提一句？
# □ ④ 建模         —— 有没有先给 baseline，再讲更复杂的方案？
# □ ⑤ 评测         —— 离线怎么测、在线怎么灰度，都说了吗？
# □ ⑥ 服务与部署   —— 延迟预算、容量、降级方案，有没有具体数字？
# □ ⑦ 迭代与风险   —— 监控什么指标、什么条件回滚，说清楚了吗？
#   ⚠️ ⑥⑦ 最容易在时间不够时被压缩成空话 —— 宁可少讲①-⑤的细节也要留时间给它们

# ══════════════════════════════════════════════════════════════════════
# C. 卡住/被追问时的应对（中 / EN）
# ══════════════════════════════════════════════════════════════════════
# 被问「为什么选这个方案」CN: 「主要是因为约束 X，取舍是我们放弃了 Y，换来了 Z。」
#                          EN: "Mainly because of constraint X; the tradeoff is we give up
#                               Y in exchange for Z."
# 被问「如果约束变了呢」  CN: 「如果 X 变成这样，我会把方案改成……，因为……」
#                          EN: "If X changes to this, I'd switch the design to ...,
#                               because ..."
# 想不出细节时           CN: 「这部分我们课程里有更细的展开，现在先给出结论，往下讲行吗？」
#                          EN: "There's a deeper rabbit hole here — let me state the
#                               conclusion and move on, is that OK?"

# ══════════════════════════════════════════════════════════════════════
# D. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 需求澄清与三层指标映射细节        -> C63 模块 01（下一站）
# · 数据系统设计（标注/泄漏/冷启动）  -> C63 模块 02
# · 建模与评测设计                    -> C63 模块 03
# · 服务、部署与容量估算              -> C63 模块 04
# · 六个完整案例演练（含 TSR 主案例） -> C63 模块 05
# · MLOps 生命周期 / 云部署 / 数据闭环 / 车端一致性的技术细节
#   -> 分别见 C37 / C48 / C58 / C60（本课引用结论，不重述）
'''
print(RECIPE)
for token in ['90 秒', '七步自检清单', 'tradeoff', 'C63 模块 01', 'C37', 'C60']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：开场脚手架 / 七步自检 / 追问应对话术 / 课程分工')

### 小结

- **system design 面试评的不是画得多复杂**，而是「需求澄清 + 取舍显式 + 能自我批判」——
  三者都是可以在 45 分钟内被面试官可靠观测到的**过程信号**，而不是「方案本身有多新颖」这种结果信号。
- **七步框架**（澄清与范围 → 成功指标 → 数据 → 建模 → 评测 → 服务与部署 → 迭代与风险）
  的价值不是「按顺序讲」，是**一份随时能核对「漏了哪一格」的清单**——真实面试会不断把你打回某一步。
- **45 分钟时间预算**里最容易翻车的不是澄清阶段拖太久，而是**⑥⑦（服务部署、迭代风险）被压缩成空话**；
  三个不可妥协的检查点是第 8 分钟（澄清完）、第 27 分钟（进入评测）、第 39 分钟（讲到服务部署）。
- **四种失败模式**（跳到模型 / 不问约束 / 不谈评测 / 不谈失败回滚）本质是同一件事：
  把一个会犯错的 ML 系统当成写对了就不会错的传统软件来设计。
- **ML system design 与传统后端 system design 的核心差异是不确定性的地位**：
  传统系统里可靠性近似取「最短板」，ML 流水线里可靠性是**链式相乘**——
  三级 95%/90%/97% 的流水线整体只剩 82.9%，比任何单级都差。这正是本课七步框架里
  ⑥⑦两步分量特别重的原因。

下一站：**模块 01 · 需求澄清与指标定义** —— 把七步框架的第①②步拆开讲透：
怎么用一份提问清单把模糊需求变成可验收规格，以及业务指标为什么总在系统上线后开始"背离"模型指标。